# Package installs

In [ ]:
! pip install -q torchcodec fasttext-numpy2 sacrebleu
! sudo apt install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 11.6 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Download raw data

### Utils

In [ ]:
import requests
import re

def download_media(url, destination_filename):
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(destination_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)
    else:
        raise Exception(
            f"Error code {response.status_code} when downloading from {url}."
        )

def get_text(tree, xpath_selector):
    result = tree.xpath(xpath_selector)
    text = result[0].text_content().strip()
    text = re.sub(r'\t', '', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r' \n', '\n', text)
    text = re.sub(r'\n ', '\n', text)
    return text

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download
import regex

NONWORD_REPLACE_STR = r"[^\p{Word}\p{Zs}]|\d"
NONWORD_REPLACE_PAT = regex.compile(NONWORD_REPLACE_STR)
SPACE_PAT = regex.compile(r"\s\s+")

def clean_line(line):
    """simple language-agnostic cleaning"""
    text = line.strip().replace("\n", " ").lower()  # remove whitespace, apply lowercase
    text = regex.sub(NONWORD_REPLACE_PAT, "", text)  # either (not a word nor a space) or (is digit)
    text = regex.sub(SPACE_PAT, " ", text)  # squeeze whitespace
    return text

model_path = hf_hub_download(repo_id="laurievb/OpenLID-v2", filename="model.bin")
model = fasttext.load_model(model_path)


In [ ]:
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata
import json

client = OpenAI(api_key=userdata.get('ELM_API_KEY'))

class TranslationResponse(BaseModel):
    translation_text: str

def get_openai_response(sys_msg, prompt, response_format, model):
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt},
        ],
        response_format=response_format,
        reasoning_effort="high",
    )
    return json.loads(completion.choices[0].message.content)

translate_system_message = lambda s, t: f"""Given some text in {s}, translate it to {t}."""

def make_translation_input(line, source_lang):
    gd_title = line["gd_title"]
    gd_text = line["gd_text"]
    en_title = line["en_title"]
    en_text = line["en_text"]

    gd_article = f"{gd_title}\n{gd_text}"
    en_article = f"{en_title}\n{en_text}"
    if source_lang == "Scottish Gaelic":
        return gd_article
    if source_lang == "English":
        return en_article

def get_translation_response(line, source_lang, target_lang, model="gpt-5.2"):
    prompt = make_translation_input(line, source_lang)
    sys_msg = translate_system_message(source_lang, target_lang)
    response = get_openai_response(sys_msg, prompt, TranslationResponse, model)
    return {f"{model}_pred_{source_lang}_to_{target_lang}": response["translation_text"]}

### Download audio and texts from LearnGaelic's Litirbheag podcast

In [ ]:
import requests
from lxml import html

def get_page_data(page_id):
    url = f"https://learngaelic.scot/litirbheag/litir.jsp?l={page_id}"
    audio_url = f"https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag{page_id.zfill(4)}.mp3"

    response = requests.get(url)

    tree = html.fromstring(response.content)

    en_text = get_text(tree, "//*[@id='enTranscript']")
    gd_text = get_text(tree, "//*[@id='gdTranscript']")
    en_title = en_text.split("\n")[0].strip()
    gd_title = gd_text.split("\n")[0].strip()
    en_text = "\n".join(en_text.split("\n")[1:]).strip()
    gd_text = "\n".join(gd_text.split("\n")[1:]).strip()
    audio_path = f"{page_id}.mp3"

    try:
        download_media(audio_url, audio_path)
    except Exception as e:
        print(f"Audio download failed at {page_id}")
        print(e)
        return {
            "page_id": page_id,
            "en_title": None,
            "gd_title": None,
            "en_text": None,
            "gd_text": None,
            "gd_audio": None,
        }

    return {
        "page_id": page_id,
        "en_title": en_title,
        "gd_title": gd_title,
        "en_text": en_text,
        "gd_text": gd_text,
        "gd_audio": audio_path,
    }

In [ ]:
from tqdm.contrib.concurrent import thread_map

page_ids = [str(i) for i in range(154, 1077)] # N.B. An Litir Bheag only starts to have parallel translation transcripts from number 154

data = thread_map(get_page_data, page_ids, max_workers=16)

  0%|          | 0/923 [00:00<?, ?it/s]

Audio download failed at 578
Error code 403 when downloading from https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag0578.mp3.
Audio download failed at 995
Error code 403 when downloading from https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag0995.mp3.
Audio download failed at 994
Error code 403 when downloading from https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag0994.mp3.
Audio download failed at 997
Error code 403 when downloading from https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag0997.mp3.
Audio download failed at 998
Error code 403 when downloading from https://s3-eu-west-1.amazonaws.com/lg-litirbheag/litirbheag0998.mp3.


In [ ]:
from datasets import Dataset

ds = Dataset.from_list(data)

In [ ]:
ds = ds.filter(
    lambda x: x is not None,
    input_columns=["gd_title"]
)

Filter:   0%|          | 0/923 [00:00<?, ? examples/s]

#### Find instances where predicted language and label language are different

This helps find nopisy examples in our data

In [ ]:
ds.filter(
    lambda x: model.predict(clean_line(x["en_text"]))[0][0].split("_")[-2] != "eng"
).to_pandas()

Filter:   0%|          | 0/918 [00:00<?, ? examples/s]

,page_id,en_title,gd_title,en_text,gd_text,gd_audio
0,288,Billy (2),Bilidh (2),Tha mi a’ leantainn leis an sgeulachd\nBilidh....,I’m continuing with the story Billy.\nBilly wa...,288.mp3
1,600,New Zealand and immigration,New Zealand agus Luchd-imrich,Tha leabhran air mo bheulaibh an-dràsta. Chaid...,Bha seachd gobhair òga aig gobhar. Latha a bha...,600.mp3
2,889,'Leum' in Place names (2),'Leum' ann an Ainmean-àite (2),Bha mi ag innse dhuibh mu ainmean-àite anns a ...,Bha mi ag innse dhuibh mu ainmean-àite anns a ...,889.mp3
3,900,The NC500 and Gaelic,An NC500 agus a’ Ghàidhlig,Different people have different opinions on th...,Tha diofar bheachdan aig diofar dhaoine air an...,900.mp3
4,914,Two anecdotes from Rannoch,Dà naidheachd à Raineach,Bha mi o chionn ghreis ann an Coille Dhubh Rai...,I was recently in the Black Wood of Rannoch or...,914.mp3
5,988,M.E.M. Donaldson (1),M.E.M. Donaldson (1),"Anns an Litir mu dheireadh, bha mi ag innse dh...","In the last Litir, I was telling you about Loc...",988.mp3
6,1060,Kintail again,Cinn Tàile A-rithist,Bha uaireigin seann iasgair ann an Cinn Tàile ...,There was at one time an old fisherman in Kint...,1060.mp3


In [ ]:
ds.filter(
    lambda x: model.predict(clean_line(x["gd_text"]))[0][0].split("_")[-2] != "gla"
).to_pandas()

Filter:   0%|          | 0/918 [00:00<?, ? examples/s]

,page_id,en_title,gd_title,en_text,gd_text,gd_audio
0,288,Billy (2),Bilidh (2),Tha mi a’ leantainn leis an sgeulachd\nBilidh....,I’m continuing with the story Billy.\nBilly wa...,288.mp3
1,585,Adhamh MacFhearghais,Adam Ferguson,I finished the Litir last week with the prover...,I finished the Litir last week with the prover...,585.mp3
2,914,Two anecdotes from Rannoch,Dà naidheachd à Raineach,Bha mi o chionn ghreis ann an Coille Dhubh Rai...,I was recently in the Black Wood of Rannoch or...,914.mp3
3,988,M.E.M. Donaldson (1),M.E.M. Donaldson (1),"Anns an Litir mu dheireadh, bha mi ag innse dh...","In the last Litir, I was telling you about Loc...",988.mp3
4,1060,Kintail again,Cinn Tàile A-rithist,Bha uaireigin seann iasgair ann an Cinn Tàile ...,There was at one time an old fisherman in Kint...,1060.mp3


#### Find instances where the AI translated text and actual translated transcript differ wildly

This helped us find instances where the article was a mis-match (Saint Mary MacKillop and MacMillan the Harpist (2))

In [ ]:
ds = ds.map(
    lambda x: get_translation_response(
        x, "Scottish Gaelic", "English",
        model="gpt-5.2"
    ),
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.TranslationResponse'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.TranslationResponse'>: __main__.TranslationResponse has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/918 [00:00<?, ? examples/s]

In [ ]:
from sacrebleu.metrics import BLEU, CHRF, TER

bleu = BLEU()

ds = ds.map(
    lambda x: {
        "sentence_bleu": bleu.sentence_score(x["gpt-5.2_pred_Scottish Gaelic_to_English"], [x["en_text"]]).score
    }
)

In [ ]:
ds.to_pandas().sort_values("sentence_bleu").iloc[:20]

,page_id,en_title,gd_title,en_text,gd_text,gd_audio,gpt-5.2_pred_Scottish Gaelic_to_English,sentence_bleu
759,914,Two anecdotes from Rannoch,Dà naidheachd à Raineach,Bha mi o chionn ghreis ann an Coille Dhubh Rai...,I was recently in the Black Wood of Rannoch or...,914.mp3,Dà naidheachd à Raineach → Two stories (or “tw...,0.000004
833,988,M.E.M. Donaldson (1),M.E.M. Donaldson (1),"Anns an Litir mu dheireadh, bha mi ag innse dh...","In the last Litir, I was telling you about Loc...",988.mp3,The text you provided is already in English (i...,0.000104
430,585,Adhamh MacFhearghais,Adam Ferguson,I finished the Litir last week with the prover...,I finished the Litir last week with the prover...,585.mp3,The text you provided is already in English (i...,0.000258
134,288,Billy (2),Bilidh (2),Tha mi a’ leantainn leis an sgeulachd\nBilidh....,I’m continuing with the story Billy.\nBilly wa...,288.mp3,The text you provided is already in English (a...,0.000610
901,1060,Kintail again,Cinn Tàile A-rithist,Bha uaireigin seann iasgair ann an Cinn Tàile ...,There was at one time an old fisherman in Kint...,1060.mp3,Kintail Again\n\nThere was at one time an old ...,0.245371
445,600,New Zealand and immigration,New Zealand agus Luchd-imrich,Tha leabhran air mo bheulaibh an-dràsta. Chaid...,Bha seachd gobhair òga aig gobhar. Latha a bha...,600.mp3,New Zealand and Immigrants\n\nA goat had seven...,0.477964
916,1075,Saint Mary MacKillop,Naomh Màiri NicIlp,Sometimes we hear false information connected ...,Tha sgìrean Gàidhealach air tìr-mòr na h-Alba ...,1075.mp3,Saint Mary MacKillop\n\nThere are Gaelic-speak...,2.678776
409,563,MacMillan the Harpist (2),Mac Gille Mhaoil na Cruit (2),I was telling you the old story from Gairloch ...,"Faisg air an t-seann chladh ann an Geàrrloch, ...",563.mp3,Mac Gille Mhaoil of the Harp (2)\n\nNear the o...,3.524980
734,889,'Leum' in Place names (2),'Leum' ann an Ainmean-àite (2),Bha mi ag innse dhuibh mu ainmean-àite anns a ...,Bha mi ag innse dhuibh mu ainmean-àite anns a ...,889.mp3,‘Leum’ in Place-names (2)\n\nI was telling you...,11.502302
560,715,A’ Ghàidhlig airson ‘twilight’,The Gaelic for 'twilight',Have you ever seen the sunset in a tropical pl...,Am faca sibh riamh dol-fodha na grèine ann an ...,715.mp3,The Gaelic for ‘twilight’\n\nHave you ever see...,20.221183


#### Filter out identified bad articles

In [ ]:
bad_article_ids = [
    715, 889, 563, 1075, 600, 1060, 288, 585, 988, 914 # These articles either have the English and Gaelic switched or have one transcript that is unrelated to the other.
]
bad_article_ids = [str(x) for x in bad_article_ids]

ds = ds.filter(
    lambda x: x["page_id"] not in bad_article_ids
)

Filter:   0%|          | 0/918 [00:00<?, ? examples/s]

#### Add article classes to dataset

In [ ]:
import email
from bs4 import BeautifulSoup

def extract_html_from_mhtml(file_path):
    with open(file_path, 'rb') as f:
        # Parse the file as a MIME message
        msg = email.message_from_binary_file(f)

    # MHTML is multipart; we need to find the part that is 'text/html'
    for part in msg.walk():
        if part.get_content_type() == 'text/html':
            # Extract and decode the payload
            html_content = part.get_payload(decode=True)
            charset = part.get_content_charset() or 'utf-8'
            return html_content.decode(charset, errors='replace')

    return "No HTML content found."

# This .mhtml file was downloaded by manually scrolling and clicking "Load More" on the following URL:
# https://learngaelic.scot/litirbheag/
html_text = extract_html_from_mhtml('/content/An Litir Bheag.mhtml')

soup = BeautifulSoup(html_text, 'html.parser')

programme_items = soup.select('ul#programmes > li')

def parse_items(li):
    class_name = li.select('h3 > span.en')[0].get_text(strip=True)
    episode_id = li.select('h4 > span.en')[0].get_text(strip=True).strip("#")
    return {
        "class_name": class_name,
        "episode_id": episode_id,
    }

class_names = [parse_items(li) for li in programme_items]

class_name_dict = {x["episode_id"]: x["class_name"] for x in class_names}

ds = ds.map(
    lambda x: {
        "subject_type": class_name_dict[x["page_id"]] if x["page_id"] in class_name_dict else None
    },
)

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

#### Save data

In [ ]:
from datasets import Audio

ds = ds.cast_column("gd_audio", Audio())

ds.select_columns(
    ['page_id', 'subject_type', 'en_title', 'gd_title', 'en_text', 'gd_text', 'gd_audio']
).push_to_hub("ptrdvn/gaelic-bench", "podcasts", split="test")

Uploading the dataset shards:   0%|          | 0/9 [00:00<?, ? shards/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          |  312kB /  504MB            

CommitInfo(commit_url='https://huggingface.co/datasets/ptrdvn/gaelic-bench/commit/bb26f444993aaef5592ac0cf6373acbcfd62fd67', commit_message='Upload dataset', commit_description='', oid='bb26f444993aaef5592ac0cf6373acbcfd62fd67', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ptrdvn/gaelic-bench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ptrdvn/gaelic-bench'), pr_revision=None, pr_num=None)

In [ ]:
import requests
from lxml import html

def get_video_page_data(page_id):
    url = f"https://learngaelic.scot/watch/watch.jsp?v={page_id}"
    response = requests.get(url)
    tree = html.fromstring(response.content)

    elements = tree.xpath(f"//source[@type='video/mp4; codecs=avc1.4D401E,mp4a.40.2']")
    if len(elements) != 1:
        print(f"No video at {page_id}")
        return {
        "page_id": page_id,
        "en_title": None,
        "gd_title": None,
        "en_text": None,
        "gd_text": None,
        "gd_video": None,
    }
    video_url = elements[0].get('src')
    video_path = f"video_{page_id}.mp4"

    try:
        download_media(video_url, video_path)
    except Exception as e:
        print(f"Video download failed at {page_id}")
        print(e)
        return {
            "page_id": page_id,
            "en_title": None,
            "gd_title": None,
            "en_text": None,
            "gd_text": None,
            "gd_video": None,
        }

    en_text = get_text(tree, "//*[@id='enTranscript']")
    gd_text = get_text(tree, "//*[@id='gdTranscript']")
    en_title = en_text.split("\n")[0].strip()
    gd_title = gd_text.split("\n")[0].strip()
    en_text = "\n".join(en_text.split("\n")[1:]).strip()
    gd_text = "\n".join(gd_text.split("\n")[1:]).strip()

    return {
        "page_id": page_id,
        "en_title": en_title,
        "gd_title": gd_title,
        "en_text": en_text,
        "gd_text": gd_text,
        "gd_video": video_path,
    }

In [ ]:
from tqdm.contrib.concurrent import thread_map

page_ids = [str(i) for i in range(1, 970)]

video_data = thread_map(get_video_page_data, page_ids, max_workers=16)

  0%|          | 0/969 [00:00<?, ?it/s]

No video at 10
No video at 20
No video at 72
No video at 76
No video at 79
No video at 80
No video at 81
No video at 82
No video at 85
No video at 96
No video at 106
No video at 111
No video at 178
No video at 187
No video at 223
No video at 249
No video at 250
No video at 251
No video at 253
No video at 255
No video at 258
No video at 260
No video at 261
No video at 263
No video at 265
No video at 266
No video at 268
No video at 270
No video at 271
No video at 273
No video at 275
No video at 277
No video at 279
No video at 281
No video at 283
No video at 285
No video at 288
No video at 290
No video at 292
No video at 294
No video at 295
No video at 298
No video at 299
No video at 301
No video at 302
No video at 303
No video at 304
No video at 305
No video at 306
No video at 307
No video at 309
No video at 310
No video at 311
No video at 312
No video at 313
No video at 314
No video at 315
No video at 317
No video at 318
No video at 320
No video at 321
No video at 323
No video at 325
No

In [ ]:
! ls *.mp4 | parallel ffmpeg -i {} -vn -acodec libmp3lame -q:a 2 {.}.mp3

In [ ]:
from datasets import Dataset

ds = Dataset.from_list(video_data).rename_columns({
    "gd_video": "file_name"
}).filter(
    lambda x: x is not None,
    input_columns=["gd_title"]
)

Filter:   0%|          | 0/968 [00:00<?, ? examples/s]

In [ ]:
ds.to_csv(
    "./gaelic_videos/metadata.csv"
)

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

4825376

In [ ]:
from huggingface_hub import HfApi, login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

api = HfApi()
repo_id = "ptrdvn/gaelic-bench-video"
folder_path = "/content/gaelic_videos"

api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

api.upload_folder(
    folder_path=folder_path,
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo=f"videos"
)

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/ptrdvn/gaelic-bench-video/commit/a8ad26aadab793b939026ff9e90e5fdc00f84791', commit_message='Upload folder using huggingface_hub', commit_description='', oid='a8ad26aadab793b939026ff9e90e5fdc00f84791', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ptrdvn/gaelic-bench-video', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ptrdvn/gaelic-bench-video'), pr_revision=None, pr_num=None)

# Generate questions

### Utils

In [ ]:
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata
import json

client = OpenAI(api_key=userdata.get('ELM_API_KEY'))

class CulturalRelevancy(BaseModel):
    cultural_relevancy_score: int

class QuestionAnswerability(BaseModel):
    en_question_answerability: int
    gd_question_answerability: int

class QuestionAnswer(BaseModel):
    question: str
    answer: str
    en_question_translation: str
    en_answer_translation: str

class QuestionsAnswers(BaseModel):
    question_answer_list: list[QuestionAnswer]

class Distractors(BaseModel):
    distractor_1: str
    distractor_2: str
    distractor_3: str
    distractor_1_en_translation: str
    distractor_2_en_translation: str
    distractor_3_en_translation: str

def get_openai_response(sys_msg, prompt, response_format, model="gpt-5.2"):
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt},
        ],
        response_format=response_format,
        reasoning_effort="high",
    )
    return json.loads(completion.choices[0].message.content)

def make_article_text(row):
    return row["gd_title"] + "\n" + row["gd_text"] + "\n\n\nEnglish translation\n" + row["en_title"] + "\n" + row["en_text"]

def make_article_text_with_questions(row):
    return row["gd_title"] + "\n" + row["gd_text"] + "\n\nQuestion:\n" + row["question"] + "\n\nAnswer:\n" + row["answer"] + "\n\n\nEnglish translation\n\n" + row["en_title"] + "\n" + row["en_text"] + "\n\nQuestion:\n" + row["en_question_translation"] + "\n\nAnswer:\n" + row["en_answer_translation"]

def make_questions_text(row):
    return "Gaelic Question:\n" + row["question"] + "\n\nEnglish Question:\n" + row["en_question_translation"]


In [ ]:
import random

def arrange_answers(row):
    answers = [row["answer"], row["distractor_1"], row["distractor_2"], row["distractor_3"]]
    en_answers = [row["en_answer_translation"], row["distractor_1_en_translation"], row["distractor_2_en_translation"], row["distractor_3_en_translation"]]

    rand_idx = random.sample(range(4), 4)
    correct_idx = rand_idx.index(0)

    ans_dict = {}

    ans_dict["correct_ans"] = chr(65 + correct_idx)

    for old_i, new_i in enumerate(rand_idx):
        ans_dict[f"answer_{chr(65 + old_i)}"] = answers[new_i]
        ans_dict[f"answer_{chr(65 + old_i)}_en"] = en_answers[new_i]

    return ans_dict

In [ ]:
distractor_gen_sys_msg = """You are a Scottish Gaelic distractor generating assistant.
Given a Gaelic article transcript and its English translation, as well as a question and answer about the content within the article, write three distractors for the answer.
The distractors should be incorrect answers to the question.
The distractors should also be plausible while remaining wrong answers.
If the correct answer is written in a particular style or format, make sure the distractors also follow this style or format.
Remember that there may be multiple correct answers to the original question, so make sure that your three distractors are all completely INCORRECT answers.
Write the distractors in Gaelic and also write an English translation for each."""

def gen_distractors(row):

    prompt = make_article_text_with_questions(row)

    return get_openai_response(
        distractor_gen_sys_msg,
        prompt,
        Distractors
    )

### Generate open questions from podcast transcripts

In [ ]:
from datasets import load_dataset

pod_ds = load_dataset("ptrdvn/gaelic-bench", "podcasts", split="test").remove_columns(
    ["gd_audio"]
)

README.md:   0%|          | 0.00/507 [00:00<?, ?B/s]

podcasts/test-00000-of-00009.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

podcasts/test-00001-of-00009.parquet:   0%|          | 0.00/395M [00:00<?, ?B/s]

podcasts/test-00002-of-00009.parquet:   0%|          | 0.00/487M [00:00<?, ?B/s]

podcasts/test-00003-of-00009.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

...

podcasts/test-00005-of-00009.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

podcasts/test-00006-of-00009.parquet:   0%|          | 0.00/443M [00:00<?, ?B/s]

podcasts/test-00007-of-00009.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

podcasts/test-00008-of-00009.parquet:   0%|          | 0.00/504M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
article_scorer_sys_msg = """You are a Scottish Gaelic article scoring assistant.
Given a Gaelic article transcript and its English translation, give a score between 1 and 5 for the cultural relevancy of the article to Gaelic culture.
If the article contains no material that relates to Gaelic culture, then give a score of 1.
If the article is full of information on important aspects of Gaelic culture, then give a score of 5.
For articles somewhere in-between these two, then give a score most fitting the content."""

def score_articles(row):

    prompt = make_article_text(row)

    return get_openai_response(
        article_scorer_sys_msg,
        prompt,
        CulturalRelevancy
    )

In [ ]:
open_qa_gen_sys_msg = """You are a Scottish Gaelic question and answer generating assistant.
Given a Gaelic article transcript and its English translation, write between 1 and 10 question in Gaelic about the content within the article.
The questions should test the answerer's knowledge of Gaelic culture in some way, using only the article as the factual basis for the question and answer.
The answers to the questions should not be easily guessed from the question.
Only include as many questions as you are able to make out of the content of the article.
Each question should be written so that it makes total sense in isolation and can be answered by someone knowledgeable on the subject without reading the article.
Make sure the questions are self contained.
You may introduce people, things, places etc. from the article in each question if that helps make the question understandable without reading the article.
Do not refer to entities contextually - always use a persons name rather than using 'the man', for example, where possible.
Each question can be as long as you would like but should be answerable in less than 10 words.
Write the questions and answers in Gaelic and also write an English translation of each."""

def gen_open_question_answer(row):

    prompt = make_article_text(row)

    try:
        return get_openai_response(
            open_qa_gen_sys_msg,
            prompt,
            QuestionsAnswers
        )
    except:
        return {
            "question_answer_list": []
        }

In [ ]:
answerability_scoring_sys_msg = """You are a Scottish Gaelic and English question scoring assistant.
Given a question in Gaelic and its English translation, give a score between 1-5 on the self-contained answerability of both questions.
Give a score of 5 if the question is a good general knowledge question, is self-contained, is not contextual dependent, and can be answered purely from using knowledge of Gaelic culture.
Give a score of 1 if the question is contextual dependent, refers to implicit information outside of general knowledge (e.g. 'the man' rather than 'Robert the Bruce'), or is otherwise badly written.
For questions somewhere in-between these two, then give a score most fitting the content.
Evaluate the question in both languages on an individual basis, evaluating the wording of the question purely in that language."""

def score_questions_answerability(row):

    prompt = make_questions_text(row)

    return get_openai_response(
        answerability_scoring_sys_msg,
        prompt,
        QuestionAnswerability,
    )

In [ ]:
pod_ds = pod_ds.map(
    score_articles,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.filter(
    lambda x: bool(
            x["cultural_relevancy_score"] >= 4
        )
)

Filter:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.map(
    gen_open_question_answer,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.QuestionAnswer'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.QuestionAnswer'>: __main__.QuestionAnswer has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/713 [00:00<?, ? examples/s]

In [ ]:
from datasets import Dataset
import pandas as pd

pod_df = pod_ds.to_pandas()
pod_df = pod_df.explode("question_answer_list").reset_index(drop=True)

pod_df = pod_df.join(pd.DataFrame(pod_df["question_answer_list"].tolist())).drop(
    "question_answer_list", axis=1
)

pod_ds = Dataset.from_pandas(pod_df)

In [ ]:
pod_ds = pod_ds.map(
    score_questions_answerability,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.QuestionAnswerability'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.QuestionAnswerability'>: __main__.QuestionAnswerability has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/6802 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.filter(
    lambda x: bool(
        x["en_question_answerability"] >= 4
    ) and bool(
        x["gd_question_answerability"] >= 4
    )
)

Filter:   0%|          | 0/6802 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.map(
    gen_distractors,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.Distractors'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.Distractors'>: __main__.Distractors has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/1087 [00:00<?, ? examples/s]

In [ ]:
pod_ds.to_pandas().to_parquet("gaelic_backup_questions.pq")

In [ ]:
pod_ds = pod_ds.map(arrange_answers)

Map:   0%|          | 0/1087 [00:00<?, ? examples/s]

In [ ]:
question_cols = ['page_id', 'question', 'correct_ans',
     'answer_A', 'answer_B', 'answer_C', 'answer_D']

pod_ds.select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "gd-podcast-cultural-questions", split="test", private=True)

pod_ds.remove_columns(
    ["question", 'answer_A', 'answer_B', 'answer_C', 'answer_D']
).rename_columns(
    {
        "en_question_translation": "question",
        "answer_A_en": "answer_A",
        "answer_B_en": "answer_B",
        "answer_C_en": "answer_C",
        "answer_D_en": "answer_D",
    }
).select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "en-podcast-cultural-questions", split="test", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  154kB /  154kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  144kB /  144kB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ptrdvn/gaelic-bench/commit/eb7fdc83c34cd6ac81f034c745b59d17e89b8f71', commit_message='Upload dataset', commit_description='', oid='eb7fdc83c34cd6ac81f034c745b59d17e89b8f71', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ptrdvn/gaelic-bench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ptrdvn/gaelic-bench'), pr_revision=None, pr_num=None)

In [ ]:
pod_ds

Dataset({
    features: ['page_id', 'en_title', 'gd_title', 'en_text', 'gd_text', 'cultural_relevancy_score', 'answer', 'en_answer_translation', 'en_question_translation', 'question', 'en_question_answerability', 'gd_question_answerability', 'distractor_1', 'distractor_2', 'distractor_3', 'distractor_1_en_translation', 'distractor_2_en_translation', 'distractor_3_en_translation', 'correct_ans', 'answer_A', 'answer_A_en', 'answer_B', 'answer_B_en', 'answer_C', 'answer_C_en', 'answer_D', 'answer_D_en'],
    num_rows: 1087
})

### Generate closed questions from podcast transcripts

In [ ]:
from datasets import load_dataset

pod_ds = load_dataset("ptrdvn/gaelic-bench", "podcasts", split="test").remove_columns(
    ["gd_audio"]
)

README.md: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
closed_qa_gen_sys_msg = """You are a Scottish Gaelic question and answer generating assistant.
Given a Gaelic article transcript and its English translation, write a question in Gaelic about the content within the article.
The question should test the answerer's comprehension of the article in some way, using only the article as the basis for the question and answer.
The answers to the question should not be easily guessed from the question.
The question should be challenging, if possible - do not make it too easy.
Each question can be as long as you would like but should be answerable in less than 10 words.
Write the question and answers in Gaelic and also write an English translation of each."""

def gen_closed_question_answer(row):

    prompt = make_article_text(row)

    try:
        return get_openai_response(
            closed_qa_gen_sys_msg,
            prompt,
            QuestionAnswer
        )
    except:
        return {
            "question": None,
            "answer": None,
            "en_question_translation": None,
            "en_answer_translation": None,
        }

In [ ]:
pod_ds = pod_ds.map(
    gen_closed_question_answer,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.QuestionAnswer'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.QuestionAnswer'>: __main__.QuestionAnswer has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.map(
    gen_distractors,
    num_proc=32
)

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.Distractors'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.Distractors'>: __main__.Distractors has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.CulturalRelevancy'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.CulturalRelevancy'>: __main__.CulturalRelevancy has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map (num_proc=32):   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
pod_ds = pod_ds.map(arrange_answers)

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
question_cols = ['page_id', 'gd_title', 'gd_text', 'question', 'correct_ans',
     'answer_A', 'answer_B', 'answer_C', 'answer_D']

pod_ds.select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "gd-podcast-comprehension-questions", split="test", private=True)

pod_ds.remove_columns(
    ["question", 'answer_A', 'answer_B', 'answer_C', 'answer_D']
).rename_columns(
    {
        "en_question_translation": "question",
        "answer_A_en": "answer_A",
        "answer_B_en": "answer_B",
        "answer_C_en": "answer_C",
        "answer_D_en": "answer_D",
    }
).select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "en-podcast-comprehension-questions", split="test", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|          | 12.0kB / 1.25MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  79%|#######9  |  979kB / 1.24MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ptrdvn/gaelic-bench/commit/977a94001c4c122ac6e61380bceec5d621114723', commit_message='Upload dataset', commit_description='', oid='977a94001c4c122ac6e61380bceec5d621114723', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ptrdvn/gaelic-bench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ptrdvn/gaelic-bench'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset

vid_ds = load_dataset(
    "ptrdvn/gaelic-bench-video",
    split="test"
).remove_columns(["videos"])

KeyboardInterrupt: 

In [ ]:
closed_qa_gen_sys_msg = """You are a Scottish Gaelic question and answer generating assistant.
Given a Gaelic article transcript and its English translation, write a question in Gaelic about the content within the article.
The question should test the answerer's comprehension of the article in some way, using only the article as the basis for the question and answer.
The answers to the question should not be easily guessed from the question.
The question should be challenging, if possible - do not make it too easy.
Each question can be as long as you would like but should be answerable in less than 10 words.
Write the question and answers in Gaelic and also write an English translation of each."""

def gen_closed_question_answer(row):

    prompt = make_article_text(row)

    try:
        return get_openai_response(
            closed_qa_gen_sys_msg,
            prompt,
            QuestionAnswer
        )
    except:
        return {
            "question": None,
            "answer": None,
            "en_question_translation": None,
            "en_answer_translation": None,
        }

In [ ]:
vid_ds = vid_ds.map(
    gen_closed_question_answer,
    num_proc=32
)

In [ ]:
vid_ds = vid_ds.map(
    gen_distractors,
    num_proc=32
)

In [ ]:
vid_ds = vid_ds.map(arrange_answers)

In [ ]:
question_cols = ['page_id', 'question', 'correct_ans',
     'answer_A', 'answer_B', 'answer_C', 'answer_D']

vid_ds.select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "gd-video-comprehension-questions", split="test", private=True)

vid_ds.remove_columns(
    ["question", 'answer_A', 'answer_B', 'answer_C', 'answer_D']
).rename_columns(
    {
        "en_question_translation": "question",
        "answer_A_en": "answer_A",
        "answer_B_en": "answer_B",
        "answer_C_en": "answer_C",
        "answer_D_en": "answer_D",
    }
).select_columns(
    question_cols
).push_to_hub("ptrdvn/gaelic-bench", "en-video-comprehension-questions", split="test", private=True)

# Upload hand-labelled data

In [ ]:
import json

with open("/content/gaelic_benchmarking_29_nov_25.jsonl", "r") as f:
    labelled_data = [json.loads(x) for x in f.readlines()]

In [ ]:
from datasets import Dataset

ds = Dataset.from_list(labelled_data).map(
    lambda x: {
        "answer_A": x["options"][0][2:].strip(),
        "answer_B": x["options"][1][2:].strip(),
        "answer_C": x["options"][2][2:].strip() if len(x["options"]) >= 3 else None,
        "answer_D": x["options"][3][2:].strip() if len(x["options"]) >= 4 else None,
        "answer_E": x["options"][4][2:].strip() if len(x["options"]) >= 5 else None,
        "correct_ans": x["correct_answer"][0].upper(),
    }
)


Map:   0%|          | 0/123 [00:00<?, ? examples/s]

In [ ]:
ds = ds.select_columns(
    ['id', 'category', 'sub_category', 'question',
     'answer_A', 'answer_B', 'answer_C',
     'answer_D', 'answer_E', 'correct_ans']
)

In [ ]:
ds.push_to_hub(
    "ptrdvn/gaelic-bench",
    "gd-manual-questions",
    split="test"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 18.0kB / 18.0kB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ptrdvn/gaelic-bench/commit/41fb3ff062bc51ad59f11730bc5a65cb7a774485', commit_message='Upload dataset', commit_description='', oid='41fb3ff062bc51ad59f11730bc5a65cb7a774485', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ptrdvn/gaelic-bench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ptrdvn/gaelic-bench'), pr_revision=None, pr_num=None)